# Curated original Colab notebook: 20 ns GROMACS MD analysis

Original source: private original Colab source (identifier withheld)

**Archival reference — not a corrected executable pipeline.** Original cell order and source code are preserved. Cached figures, data outputs, execution history and incidental Colab metadata are removed. No MD analysis was rerun by importing this notebook.

**Review notes:** scripts depend on hard-coded `/content/drive/MyDrive/gromacs` paths and a GROMACS executable in a separate Drive folder. The code labels center-of-geometry separation as COM, without periodic minimum-image handling; distance spikes can result. H-bond atom selections and fixed frame-to-time conversions need verification. The source loop uses `end_frame=200` even though the extracted trajectory contains 201 frames. Consult `docs/MD_ORIGINAL_NOTEBOOK_AUDIT.md` and `results/md/README.md` before reuse.


# installing libraries part 1 (when it asks to restart runtime, click on restart)

* MDAnalysis: https://www.mdanalysis.org/
* mdakit-sasa: https://mdakit-sasa.readthedocs.io/en/latest/
* plotly: https://plotly.com/python/getting-started/

In [ ]:
!pip install --upgrade MDAnalysis
!pip install mdakit-sasa
!pip install MDAnalysisTests
!pip install plotly --upgrade
!pip install kaleido --upgrade

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# installing libraries part 2 (when it asks to restart runtime, click on restart)

* rdkit: https://www.rdkit.org/docs/GettingStartedInPython.html
* prolif: https://prolif.readthedocs.io/en/stable/


In [ ]:
!pip install rdkit prolif
!echo y|plotly_get_chrome
!sudo apt update && sudo apt-get install libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2

# Define important things

In [ ]:
#@title change accordingly

#@markdown * Change the DIR_sys1 variable to point to the system 1 folder (minimization and equilibration analysis)
#@markdown * Change the DIR_sys2 variable to point to the system 2 folder (production analysis)
#@markdown * **DO NOT ADD ANY SYMBOL OR SPACE IN THE NAME OF THE FOLDER**

#@markdown * **Since you dont have these folders, you cant run these analyses on these files**
#@markdown * **In your case you will do these analyses ONLY when you perform the MD simulation in your project**

import plotly.graph_objects as go
import plotly.io as pio
import warnings



DIR_sys1 = '/content/drive/MyDrive/gromacs' #@param {type:"string"}
DIR_sys2 = '/content/drive/MyDrive/gromacs' #@param {type:"string"}

# Ignore all DeprecationWarnings
warnings.filterwarnings("ignore")

pio.default_format = "svg"
pio.default_width = 1200
pio.default_height = 600
pio.default_scale = 5

def update_fig(fig, titlename, xname, yname, y=.9, font_size = 32):
    fig.update_layout(
        title={
           'text': titlename,
           'y':y,
           'x':0.5,
           'xanchor': 'center',
           'yanchor': 'top'},
        xaxis_title= xname,
        yaxis_title= yname,
        font_family="Times New Roman",
        font_color="black",
        font_size = font_size,
        title_font_family="Times New Roman",
        title_font_color="black",
        )
    return fig

def parse_data(data):
    x_axis = []
    y_axis = []
    x_title = None
    y_title = None
    title = None
    for l in data.splitlines():
        if 'xaxis  label' in l:
            x_title = l.split('\"')[1]
            continue
        if 'yaxis  label' in l:
            y_title = l.split('\"')[1]
            continue
        if 'title' in l:
            title = l.split('\"')[1]
            continue
        elems = l.strip().split()
        if len(elems) == 2:
            try:
                temp = float(elems[0])
                x_axis.append(float(elems[0]))
                y_axis.append(float(elems[1]))
            except:
                continue
    return title, x_axis, y_axis, x_title, y_title

# Minimization step analysis
* change the `<PATH>` to the DIR_sys1 folder

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/gromacs
gmx energy -f step4.0_minimization.edr -o potential.xvg

In [ ]:
with open(f'{DIR_sys1}/potential.xvg') as f:
  data = f.read()
title, x_axis, y_axis, x_title, y_title = parse_data(data)

In [ ]:
fig = go.Figure(go.Scatter(x=x_axis, y=y_axis))
fig = update_fig(fig, 'Minimization', 'Minimization Steps', 'Energy (KJ/mol)')
fig.write_image(f'{DIR_sys1}/Minimization.svg', width=1200, height=600, scale = 5)
fig.show()

# Equilibration steps analysis
* change the `<PATH>` to the DIR_sys1 folder

## NVT

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/gromacs

gmx energy -f step4.1_equilibration.edr -o temperature.xvg

In [ ]:
with open(f'{DIR_sys1}/temperature.xvg') as f:
  data = f.read()
title, x_axis, y_axis, x_title, y_title = parse_data(data)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(go.Scatter(x=x_axis, y=y_axis))
fig = update_fig(fig, 'NVT Equilibration', 'Equilibration Steps', 'Temperature (Kelvin)')
fig.write_image(f'{DIR_sys1}/NVT Equilibration.svg', width=1200, height=600, scale = 5)
fig.show()

## NPT

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/gromacs

gmx energy -f step4.2_equilibration.edr -o pressure.xvg

In [ ]:
with open(f'{DIR_sys1}/pressure.xvg') as f:
  data = f.read()
title, x_axis, y_axis, x_title, y_title = parse_data(data)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(go.Scatter(x=x_axis, y=y_axis))
fig = update_fig(fig, 'NPT Equilibration', 'Equilibration Steps', 'Pressure (bar)')
fig.write_image(f'{DIR_sys1}/NPT Equilibration.svg', width=1200, height=600, scale = 5)
fig.show()

# Preparing the trajectory for analysis and visualizing it

* change the `<PATH>` to the DIR_sys2 folder
* make a new .ndx file containing the complex in one group
* combine all .xtc files in one file and output the complex ONLY
* create a new .tpr corresponding to the new .xtc file containing the complex ONLY
* remove the effect of PBC by centering on the receptor


In [ ]:
%%shell
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
cd /content/drive/MyDrive/gromacs

gmx make_ndx -f step4.0_minimization.gro -o new_index.ndx
gmx trjcat -f step5*.xtc -n new_index.ndx -o all.xtc

In [ ]:
%%shell
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
cd /content/drive/MyDrive/gromacs

gmx convert-tpr -s step5_production.tpr -n new_index.ndx -o step5_production_SOLU.tpr
gmx trjconv -f all.xtc -s step5_production_SOLU.tpr -n new_index.ndx -pbc mol -center -ur compact -o all_noPBC.xtc

# Basics analysis

## importing libraries (RUN WITHOUT MODIFICATIONS)

In [ ]:
# https://www.mdanalysis.org/

import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
from MDAnalysis.topology.guessers import guess_types
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
from mdakit_sasa.analysis.sasaanalysis import SASAAnalysis
import plotly.graph_objects as go
import pandas as pd
import numpy as np

mda.__version__

## Loading the trajectory (RUN WITHOUT MODIFICATIONS)


In [ ]:
import MDAnalysis as mda
from MDAnalysis.topology.guessers import guess_types

XTC = DIR_sys2+'/all_noPBC.xtc'
TPR = DIR_sys2+'/step5_production_SOLU.tpr'
uni = mda.Universe(TPR, XTC)

guessed_elements = guess_types(uni.atoms.names)
uni.add_TopologyAttr('elements', guessed_elements)

print(f'You have {len(uni.trajectory)} frames')
# print(uni.residues.resids)

## 1D RMSD (RUN WITHOUT MODIFICATIONS)

* Note: You can modify the code if you know whatyou want to measure and how to measure it in MDAnalysis

In [ ]:
import time as t
start = t.time()
domain = 'backbone' # the same atom selection language in MDA (https://userguide.mdanalysis.org/stable/selections.html)
R = rms.RMSD(uni,  # universe to align
             uni,  # reference universe or atomgroup
             select='backbone',  # group to superimpose and calculate RMSD
             groupselections=['resname UNL'],  # groups for RMSD
             ref_frame=0)  # frame index of the reference
R.run()
end = t.time()
(end-start)/60

In [ ]:
res = R.results.rmsd # 0: steps, 1: time, 2: backbone, 3 and above: groupselection in the same order
fig = go.Figure()

fig.add_trace(go.Scatter(x= res[:, 1]/1000, y = res[:, 2], name= 'Backbone'))
fig.add_trace(go.Scatter(x= res[:, 1]/1000, y = res[:, 3], name= 'Ligand'))

fig.update_layout(showlegend=True,
                  legend=dict(
                              y=-0.25,
                              xanchor="center",
                              x=0.5,
                              orientation='h'
                              )
                  )
fig = update_fig(fig, '1D RMSD', 'Time (ns)', 'RMSD (Å)')
fig.write_image(f'{DIR_sys2}/1D RMSD.svg', width=1200, height=600, scale = 5)
fig.write_html(f"{DIR_sys2}/1D RMSD.html")

fig.show()


## 2D RMSD  (RUN WITHOUT MODIFICATIONS)

* Note: You can modify the code if you know whatyou want to measure and how to measure it in MDAnalysis

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import diffusionmap, align, rms
import numpy as np
import plotly.graph_objects as go

# 1. Prepare for 2D RMSD calculation
# align the prajectory to itself
aligner = align.AlignTraj(uni, uni, select='backbone',
                          in_memory=True).run()

# 2. Perform the 2D RMSD calculation
print("Calculating 2D RMSD matrix... (this may take a moment)")
matrix = diffusionmap.DistanceMatrix(uni, select='name CA').run()
print('finished')


In [ ]:
# 3. Plot the 2D RMSD matrix using Plotly
# Get the simulation time in nanoseconds for axis labels
time_ns = np.arange(0, len(uni.trajectory)//10+0.1, .1)

print("Generating plot...")
fig = go.Figure(data=go.Heatmap(z = matrix.results.dist_matrix,
                                x = time_ns,
                                y = time_ns,
                                colorscale='Plasma',  # Virdis
                                colorbar_title= 'RMSD (Å)'
                                ))
# 5. Update the layout to add titles and improve appearance
fig = update_fig(fig, 'Backbone 2D RMSD', 'Time (ns)', 'Time (ns)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.write_image(f'{DIR_sys2}/2D RMSD.svg', width=1200, height=600, scale = 5)
fig.write_html(f"{DIR_sys2}/2D RMSD.html")

fig.show()

## RMSF (RUN WITHOUT MODIFICATIONS)

* Note: You can modify the code if you know whatyou want to measure and how to measure it in MDAnalysis

In [ ]:
start = t.time()

uni.trajectory[0] # reach the first frame
sele = "protein and name CA"
ref = uni.select_atoms(sele) # select the ref atoms
aligner = align.AlignTraj(uni,
                          ref,
                          select=sele,
                          in_memory=True).run()
c_alphas = uni.select_atoms(sele)
R = rms.RMSF(c_alphas).run()
end = t.time()
(end-start)/60

In [ ]:
#R.results.rmsf.shape, c_alphas.resids.shape

ids, rmsf =  c_alphas.resids, R.results.rmsf
fig = go.Figure()

fig.add_trace(go.Scatter(x= ids, y = rmsf))

fig = update_fig(fig, 'Cα RMSF', 'Amino acid number', 'RMSF (Å)')

fig.write_image(f'{DIR_sys2}/Cα RMSF.svg', width=1200, height=600, scale = 5)
fig.write_html(f"{DIR_sys2}/Cα RMSF.html")

fig.show()

## Hbonds between ligand and protein (RUN WITHOUT MODIFICATIONS)

* Note: You can modify the code if you know whatyou want to measure and how to measure it in MDAnalysis

In [ ]:
import time as t
start = t.time()
hbonds = HydrogenBondAnalysis(
    universe=uni,
    donors_sel='name S* or name N* or name O* or name F*', #polar atoms
    hydrogens_sel='name H*',
    acceptors_sel='name S* or name N* or name O* or name F*',
    between = ["protein", "resname UNL"],
    d_a_cutoff=3.0,
    d_h_a_angle_cutoff=160,
    update_selections=True
).run() # all frames
      #start=0,
      #stop=5001,
      #step=1)
end = t.time()

# [frame, donor_index, hydrogen_index, acceptor_index, DA_distance, DHA_angle]
df = pd.DataFrame(hbonds.results.hbonds, columns=['frame', 'donor_index', 'hydrogen_index', 'acceptor_index', 'DA_distance', 'DHA_angle'])
df.head()
count = df.groupby('frame').count()
h_bonds = count.donor_index
new = pd.DataFrame({'donor_index':[0]*len(uni.trajectory)})
new.index.name = 'frames'
new.index = new.index.astype(float)

final = pd.concat([new, h_bonds], axis=1).sum(axis=1)
final.index = final.index/10 # convert frames to ns
final = pd.DataFrame(final, columns=['values'])
(end-start)/60 , ' in min'

In [ ]:
# help(HydrogenBondAnalysis)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x= final.index, y = final.values[:,0]))
fig = update_fig(fig, 'Change in H-bonds number between the protein and ligand', 'Time (ns)', 'No. H-bonds', y=.94)
fig.update_layout(yaxis_range=[0, final.values[:,0].max() + 3])


fig.write_image(f'{DIR_sys2}/H-bonds.svg', width=1200, height=600, scale = 5)
fig.write_html(f"{DIR_sys2}/H-bonds.html")

fig.show()

## SASA (RUN WITHOUT MODIFICATIONS)

* Note: You can modify the code if you know whatyou want to measure and how to measure it in MDAnalysis

In [ ]:
s= t.time()
analysis = SASAAnalysis(uni, select='protein')
analysis.run(start = 0,
             stop = 500,
             step = 1,
               )
e= t.time()
print((e-s)/60)
# print(analysis.results.total_area)


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x= np.arange(len(analysis.results.total_area))/10,
                         y = analysis.results.total_area))

fig = update_fig(fig, 'SASA', 'Time (ns)', 'SASA (Å<sup>2</sup>)')
# fig.update_layout(yaxis_range=[0, analysis.results.total_area.max()])

fig.write_image(f'{DIR_sys2}/SASA.svg', width=1200, height=600, scale = 5)
fig.write_html(f"{DIR_sys2}/SASA.html")

fig.show()

## RoG (RUN WITHOUT MODIFICATIONS)

* Note: You can modify the code if you know whatyou want to measure and how to measure it in MDAnalysis

In [ ]:
protein = uni.select_atoms('protein')
rog = []
for ts in uni.trajectory:
    rad = protein.radius_of_gyration()
    rog.append(rad)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x= np.arange(len(rog))/10,
                         y = rog))
fig = update_fig(fig, 'RoG', 'Time (ns)', 'RoG (Å)')
# fig.update_layout(yaxis_range=[0, max(rog)])
fig.write_image(f'{DIR_sys2}/RoG.svg', width=1200, height=600, scale = 5)
fig.write_html(f"{DIR_sys2}/RoG.html")

fig.show()

## change in center of mass (COM) separation between ligand and protein (RUN WITHOUT MODIFICATIONS)

* Note: You can modify the code if you know whatyou want to measure and how to measure it in MDAnalysis

In [ ]:
protein = uni.select_atoms('protein', updating = True)
lig = uni.select_atoms('resname UNL', updating = True)
separations = []

for ts in uni.trajectory:
    comcom_dist = lig.center_of_geometry() - protein.center_of_geometry()
    comcom_dist = abs(np.linalg.norm(comcom_dist))
    separations.append(comcom_dist)



In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x= np.arange(len(separations))/10,
                         y = separations))

fig = update_fig(fig, 'Center of Mass separation between the protein and ligand', 'Time (ns)', 'Distance (Å)')
fig.update_layout(yaxis=dict(range=[0, max(separations)+3]))

fig.write_image(f'{DIR_sys2}/COMCOM.svg', width=1200, height=600, scale = 5)
fig.write_html(f"{DIR_sys2}/COMCOM.html")
fig.show()

## ProLIF (expand the cell to know what you can change)

In [ ]:
import prolif as plf
import time as t
import MDAnalysis as mda
from MDAnalysis.topology.guessers import guess_types
import seaborn as sns
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
plf.__version__

In [ ]:
# create selections for the ligand and protein
lig = uni.atoms.select_atoms("resname UNL")
prot = uni.atoms.select_atoms("protein")

prot.chainIDs = np.array(len(prot.chainIDs)*['A'])
lig.chainIDs = np.array(len(lig.chainIDs)*['X'])

# create a molecule from the MDAnalysis selection
lmol = plf.Molecule.from_mda(lig)
plf.display_residues(lmol, size=(400, 200))

In [ ]:
# ?fp.run

* You can modify the `end_frame` value from 1 to the number of frames of your system

In [ ]:
end_frame = 200

# use default interactions
fp = plf.Fingerprint()
start = t.time()
# run on a slice of frames from begining to end with a step of 1
fp.run(uni.trajectory[:end_frame], lig, prot,
       n_jobs=None, residues=None, progress=True) # detect amino acids close to the ligand (6 A)
end = t.time()

print(f'it took {(end-start)/60} mins')

fp.to_pickle(DIR_sys2+'/fingerprint.pkl')


In [ ]:
# Reload to continue analysis
# fp = plf.Fingerprint.from_pickle(DIR_sys2+'/fingerprint.pkl')

* You can modify the `prob` value from 0 to 1.0
* This shows the 2D interaction plot

In [ ]:
prob=0.85

fp.plot_lignetwork(lmol, threshold=prob)


* This shows the occurence proportion for each interaction above the `prob`

In [ ]:
# to show the occurence of interaction types for each amino acid
df = fp.to_dataframe()
# calculte the occurence of each interaction on the trajectory
occ = df.mean()
# restrict to the frequent ones
inter = occ.loc[occ >= prob]
tri = inter.reset_index()
idxs = [(lig, aa, inter) for lig, aa, inter in zip(tri.ligand, tri.protein, tri.interaction)]
selected_columns = df.loc[:, idxs]
selected_columns.index = selected_columns.index/10

print(tri)

* This shows a the appearance of interactions in each frame

In [ ]:
import matplotlib.pyplot as plt
from prolif.plotting.barcode import Barcode

y_size = int(len(tri)*0.7)
fig = Barcode(selected_columns).display(
    figsize=(8,y_size),
    xlabel='Time (ns)',
    dpi = 700
)
plt.savefig(DIR_sys2+'/Prolif.tiff')

In [ ]:
# def remove_title(x):
#   x = list(x)
#   x[0] = 'LIG'
#   x[1] = x[1].replace('.Title', '')
#   return tuple(x)
# selected_columns.columns = selected_columns.columns.map(remove_title)
# selected_columns

## Compress the figures and download a .gz file containing all the imges

In [ ]:
import os
os.system(f'tar -zcf {DIR_sys2}/Basic_Analysis.tar.gz {DIR_sys2}/*.html {DIR_sys2}/*.svg {DIR_sys2}/*.tiff')
# os.system(f'rm  {DIR_sys2}/*.svg {DIR_sys2}/*.tiff {DIR_sys2}/*.html')